# STL 分解（Seasonal-Trend decomposition using LOESS，1990）

- **核心原理**：拆成趨勢、季節、殘差
- **變體**：古典分解（移動平均，1920s）、X-11／X-13（官方統計）、MSTL（多重季節）
- **推薦資料集**：AirPassengers（同第 1、2 章：train 1949–1958、test 1959–1960）
- **實作**：✋ 手刻古典分解 + LOESS（STL 的積木）；STL 本身用 🧰 statsmodels
- **文獻**：Cleveland, Cleveland, McRae & Terpenning (1990), *STL: A Seasonal-Trend Decomposition Procedure Based on Loess*；fpp3 第 3 章

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

前面幾章都在「預測」；分解的主要目的是**理解與前處理**：

$$y_t = T_t + S_t + R_t \quad\text{（加法）}\qquad\text{或}\qquad y_t = T_t \times S_t \times R_t \quad\text{（乘法；取 log 就變加法）}$$

| | 第 1 章 指數平滑 | 本章 分解 |
| --- | --- | --- |
| 趨勢、季節 | 狀態 $\ell_t + b_t$、$s_t$，**只用過去**一步一步更新（可以即時預測） | $T_t$、$S_t$，**用前後的資料**一次估計（中心化的平滑） |
| 用途 | 預測 | 看結構、季節調整（失業率、GDP 的「季調後」數字）、異常偵測、當預測的前處理 |
| 平滑工具 | 指數加權平均 | 中心化移動平均（古典）、**LOESS**（局部加權迴歸，STL） |

一句話：**STL 反覆用 LOESS 平滑「每個月的子序列」得到季節、平滑「去掉季節後的序列」得到趨勢，直到收斂 —— 比古典分解有彈性（季節可以慢慢變）、對異常值穩健。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A–3D、4、5 節開頭另有各自的符號表或名詞表。

**資料**：同第 1、2 章。分解在 **log 尺度**上做（季節波動隨水準變大 → 乘法 → log 之後加法）；評估預測時 exp 回原尺度。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $y_t$、$x_t = \log y_t$ | 原始、log | |
| $m$ | 季節週期 | 12 |
| $T_t$、$S_t$、$R_t$ | 趨勢、季節、殘差（remainder） | |
| $\bar S_j$ | 第 $j$ 個月的季節指數（$j = 0..m-1$） | |
| $x_t - S_t$ | **季節調整後**的序列 | |
| $q$、frac | LOESS 每個點用的鄰居數、鄰居佔全部的比例 | |
| $F_T$、$F_S$ | 趨勢強度、季節強度（0～1） | |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第 1 章 | 指數平滑 | 季節性 naive test MASE 2.49；CV 選出「log + Holt-Winters 加法」：test MASE 0.66；滾動起點 CV（起點 84、96、108，各 12 步）。 |
| 第 2 章 | ARIMA | 航空模型 SARIMA(0,1,1)(0,1,1)$_{12}$（log）：test MASE 1.38；兩次差分的隨機漫步 2.43；one-standard-error rule 在只有 3 個 fold 時太寬鬆。 |
| 第一部分第 2 章 | 線性回歸 | 加權最小平方：$\min\sum w_i(y_i - \beta_0 - \beta_1 x_i)^2$ → 把每一列乘上 $\sqrt{w_i}$ 再做 OLS。LOESS = 每個點各做一次。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 中心化移動平均**：$m$ 是偶數（12）時，「前 6 後 5」不對稱 → 用 **2×12 MA**：先做 12 期平均、再做 2 期平均，等於權重 $(\frac12, 1, 1, \dots, 1, \frac12)/12$（13 個點、對稱）。每個月剛好權重 $1/12$（頭尾兩個同月份各 $1/24$）→ 固定的季節形狀被平均掉，留下趨勢。頭尾各 6 個點算不出來（NaN）。

**2. 季節指數**：去趨勢 $x_t - T_t$，把同一個月的值平均 → $\bar S_j$；再減掉 $\bar S$ 的平均讓 12 個加總為 0（加法）。乘法版：$y_t / T_t$ 平均、再除以平均讓它們的平均為 1。

**3. LOESS（局部加權線性迴歸）**：要估計 $x_0$ 處的平滑值：取最近的 $q$ 個點，距離 $d_i$，最遠的那個距離 $h$；權重 tricube $w_i = (1 - (d_i/h)^3)^3$（$d_i \ge h$ 的權重 0）；做加權線性迴歸 $\min\sum w_i(y_i - a - b(x_i - x_0))^2$，平滑值 $= a$。例：$d = (0, 1, 2)$、$h = 2$ → $w = (1, 0.670, 0)$。

**4. 變異數分解與強度**：$F_T = \max\big(0, 1 - \frac{\text{Var}(R)}{\text{Var}(T + R)}\big)$、$F_S = \max\big(0, 1 - \frac{\text{Var}(R)}{\text{Var}(S + R)}\big)$（Wang, Smith & Hyndman 2006）：去掉趨勢（季節）之後，剩下的變化有多少是殘差。接近 1 → 很強。

#### 程式

**5. `np.convolve(x, w, mode="valid")`**：滑動加權和；長度 $n - \text{len}(w) + 1$。對稱的 $w$ 不用翻轉。
**6. `np.arange(n) % m`**：每個點是第幾個月（0 = 一月，因為資料從一月開始）。
**7. `np.nanmean(a[idx == j])`**：忽略 NaN 的平均。

---
## 1. 資料與探索性資料分析（EDA）

第 1 章已經看過趨勢、季節與「季節波動隨水準變大」。這裡只用一張圖確認：**加法分解**在原尺度與 log 尺度的殘差各長什麼樣子（用 statsmodels 的古典分解，3A 會自己寫一次）。

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from ml_dl_llm.data import load_air_passengers, split_series

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
y = load_air_passengers()
if isinstance(y.index, pd.PeriodIndex):
    y.index = y.index.to_timestamp()
y = y.astype(float).rename("passengers")
y_train, y_test = split_series(y, test_size=24)
M, H = 12, len(y_test)
x_train = np.log(y_train)

fig, axes = plt.subplots(2, 2, figsize=(16, 6))
for col, (name, s) in enumerate([("raw y", y_train), ("log y", x_train)]):
    dec = seasonal_decompose(s, model="additive", period=M)
    s.plot(ax=axes[0, col], color="k", lw=1, label=name)
    dec.trend.plot(ax=axes[0, col], label="trend")
    axes[0, col].legend()
    dec.resid.plot(ax=axes[1, col], color="tab:red")
    axes[1, col].axhline(0, color="k", lw=0.5)
    axes[1, col].set_title(f"additive remainder on {name}")
plt.tight_layout()
plt.show()
for name, s in [("raw y", y_train), ("log y", x_train)]:
    r = seasonal_decompose(s, model="additive", period=M).resid.dropna()
    half = len(r) // 2
    print(f"{name}：殘差 std 前半 {r.iloc[:half].std():.4f}、後半 {r.iloc[half:].std():.4f}")

In [ ]:
# TODO 1（問答）EDA
#
# (1) 原尺度的加法分解：殘差在前半、後半的 std 差多少？殘差圖裡還看得到季節的形狀嗎？為什麼？（提示：季節的幅度在變大，但加法分解只估一組固定的季節指數）
# (2) log 尺度：殘差的 std 前後一致嗎？殘差還有規律的形狀嗎？
# (3) 古典分解的趨勢頭尾各少了幾個點？為什麼？（前置複習 1）這對「用分解來預測」有什麼影響？
# 答：

---
## 2. Baseline

| Baseline | test MASE |
| --- | --- |
| 季節性 naive | 2.49（第 1 章） |
| （第 1 章）log + Holt-Winters 加法（手寫） | 0.66 |
| （第 2 章）航空模型 SARIMA | 1.38 |

第 6 節會把「用分解來預測」的方法加進同一張表（`results/air_passengers.csv`）。這裡先算季節性 naive 確認。

In [ ]:
from ml_dl_llm.metrics import forecast_metrics

snaive = y_train.to_numpy()[len(y_train) - M + (np.arange(H) % M)]
test_table = pd.DataFrame({"Seasonal naive": forecast_metrics(y_test, snaive, y_train, season=M)}).T
display(test_table)

In [ ]:
# TODO 2（問答）先寫下預測：「把 log y 分解成趨勢 + 季節，季節用去年的、趨勢用一條直線延伸」這種預測，會比季節性 naive 好嗎？比 Holt-Winters 呢？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 古典分解：2×$m$ 移動平均（趨勢）、季節指數、殘差 | 3、4、5 |
| 3B | LOESS：STL 的積木 | 6 |
| 3C | 對照 statsmodels（`seasonal_decompose`、`lowess`） | 7 |

### 3A. 古典分解

1. **趨勢**：$T_t$ = 2×$m$ 移動平均（前置複習 1），頭尾各 $m/2$ 個 NaN。
2. **季節**：$\bar S_j$ = 「$x_t - T_t$ 在第 $j$ 個月的平均」，再置中（加總為 0）；$S_t = \bar S_{t \bmod m}$。
3. **殘差**：$R_t = x_t - T_t - S_t$。

（乘法版：$y_t / T_t$、平均為 1、$R_t = y_t / (T_t S_t)$；`model="mul"`。）

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $x$ | 序列 | $(n,)$ | `x` |
| $w$ | 2×$m$ MA 的權重 | $(m + 1,)$ | `w` |
| $T$ | 趨勢（頭尾 NaN） | $(n,)$ | `trend` |
| `month` | 每個點是第幾個月 | $(n,)$ | `np.arange(n) % m` |
| $\bar S$ | 季節指數 | $(m,)$ | `s_idx` |

**下標公式**：
- TODO 3：`w = np.r_[0.5, np.ones(m − 1), 0.5] / m`；`trend[m//2 : n − m//2] = np.convolve(x, w, mode="valid")`。
- TODO 4（加法）：`det = x − trend`；`s_idx = [np.nanmean(det[month == j]) for j in range(m)]`；`s_idx −= s_idx.mean()`。乘法：`det = x / trend`、`s_idx /= s_idx.mean()`。
- TODO 5：`resid = x − trend − seasonal`（乘法：`x / (trend * seasonal)`）。

#### 最小例子：TODO 3、4、5（$m = 4$）

$x_t = 10 + t + s_{t \bmod 4}$，$s = (2, -2, 1, -1)$，$t = 0..7$ → $x = (12, 9, 13, 12, 16, 13, 17, 16)$。

1. **正確答案**：2×4 MA 的權重 $(0.5, 1, 1, 1, 0.5)/4$；$T_2 = (0.5 \cdot 12 + 9 + 13 + 12 + 0.5 \cdot 16)/4 = 12$（= $10 + 2$，季節剛好被平均掉）；$T_{2..5} = (12, 13, 14, 15)$，頭尾各 2 個 NaN。去趨勢後第 2..5 點 = $(1, -1, 2, -2)$ → 依月份排回 $\bar S = (2, -2, 1, -1)$（剛好還原）；殘差 = 0。
2. **積木**：`np.convolve(x, w, mode="valid")` 的長度與對應的位置。
3. **留給你**：TODO 3、4、5。

In [ ]:
# 最小例子：TODO 3、4、5 —— 2. 積木
x_mini = np.array([12.0, 9.0, 13.0, 12.0, 16.0, 13.0, 17.0, 16.0])
w_mini = np.r_[0.5, np.ones(3), 0.5] / 4
print("權重：", w_mini, "；convolve(valid)：", np.convolve(x_mini, w_mini, mode="valid"), "← 對應 t = 2..5")
# 寫完 TODO 3–5、執行過下一格之後，取消下面的註解來對照（趨勢 [nan nan 12 13 14 15 nan nan]、季節指數 [2 −2 1 −1]、殘差 0）：
# d_mini = classical_decompose(x_mini, m=4); print(d_mini["trend"], d_mini["s_idx"], np.nanmax(np.abs(d_mini["resid"])))

In [ ]:
def classical_decompose(x: np.ndarray, m: int = M, model: str = "add") -> dict:
    """古典分解。回傳 {"trend", "seasonal", "resid" (n,)，"s_idx" (m,)}。資料的第 0 個點是「第 0 個月」。"""
    x = np.asarray(x, dtype=float)
    n = len(x)
    trend = np.full(n, np.nan)
    # TODO 3（程式）：2×m 移動平均（m 為偶數）
    raise NotImplementedError("TODO 3：移動平均")
    month = np.arange(n) % m
    # TODO 4（程式）：季節指數 s_idx (m,)（加法：置中為 0；乘法：平均為 1）
    raise NotImplementedError("TODO 4：季節指數")
    seasonal = s_idx[month]
    # TODO 5（程式）：殘差
    raise NotImplementedError("TODO 5：殘差")
    return {"trend": trend, "seasonal": seasonal, "resid": resid, "s_idx": s_idx}

### 3B. LOESS

STL 的每一步都是 LOESS（前置複習 3）：對每個點，找最近的 $q = \lfloor\text{frac}\cdot n\rfloor$ 個鄰居、tricube 權重、加權線性迴歸，取迴歸線在這個點的值。

- frac 小 → 只看附近 → 跟得緊、比較抖；frac 大 → 看很遠 → 平滑、可能抹平轉折。
- **穩健版**（STL 的 `robust=True`）：先做一次，殘差大的點在下一輪降低權重（bisquare），重複幾次 → 異常值的影響變小。

#### 符號表（3B）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| $x_0$ | 要估計的位置 | `xv[i]` |
| $d$ | 所有點到 $x_0$ 的距離 | `d` |
| $h$ | 第 $q$ 近的距離 | `h = np.sort(d)[q − 1]` |
| $w$ | tricube 權重 | `w` |
| $A$ | 設計矩陣：常數、$x - x_0$ | `A` |

**下標公式**：
- TODO 6：`w = np.clip(1 − (d / h) ** 3, 0, None) ** 3`；加權最小平方：`sw = np.sqrt(w)`，`beta = np.linalg.lstsq(A * sw[:, None], yv * sw, rcond=None)[0]`；平滑值 `= beta[0]`（截距 = 在 $x_0$ 的值，因為 $A$ 的第二欄是 $x - x_0$）。

#### 最小例子：TODO 6

$x = (0, 1, 2, 3, 4)$、$y = (0, 1, 4, 9, 16)$（$y = x^2$），在 $x_0 = 2$、$q = 5$（全部的點）：$h = 2$ → $w = (0, 0.670, 1, 0.670, 0)$；只有 $x = 1, 2, 3$ 有權重、左右對稱 → 斜率 6、截距 = 加權平均 $\frac{0.670 \cdot 1 + 4 + 0.670 \cdot 9}{2.340} \approx 4.5726$（比真值 4 高：曲線往上彎，直線近似會「偏向」兩邊）。

In [ ]:
def loess(xv: np.ndarray, yv: np.ndarray, frac: float) -> np.ndarray:
    """局部加權線性迴歸（tricube、不穩健、在每一個 xv 上估計）。q = int(frac · n)，同 statsmodels。"""
    n = len(xv)
    q = int(frac * n + 1e-5)
    out = np.zeros(n)
    for i in range(n):
        d = np.abs(xv - xv[i])
        h = np.sort(d)[q - 1]
        A = np.column_stack([np.ones(n), xv - xv[i]])
        # TODO 6（程式）：tricube 權重、加權最小平方、取截距（見下標公式）
        raise NotImplementedError("TODO 6：LOESS")
    return out


# 寫完 TODO 6 之後，取消下面的註解來對照（中間那個應為 4.5726）：
# print(loess(np.arange(5.0), np.arange(5.0) ** 2, frac=1.0).round(4))

### 3C. 對照 statsmodels；實驗

- `seasonal_decompose(x, model="additive"／"multiplicative", period=12)`：古典分解，趨勢、季節、殘差應該**完全一樣**。
- `statsmodels.nonparametric.smoothers_lowess.lowess(y, x, frac, it=0)`：`it=0` = 不做穩健的迭代，應該跟手寫的一樣。

**實驗**：
1. LOESS 的 frac：0.05、0.15、0.4 在 log y 上的平滑結果；
2. 古典分解的季節指數（log 尺度）→ 換回「比例」：每個月比全年平均高幾 %？

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

for model, sm_model, series in [("add", "additive", x_train), ("mul", "multiplicative", y_train)]:
    mine = classical_decompose(series.to_numpy(), m=M, model=model)
    sm = seasonal_decompose(series, model=sm_model, period=M)
    print(f"✅ {sm_model:14s}：趨勢差 {np.nanmax(np.abs(mine['trend'] - sm.trend.to_numpy())):.1e}；季節差 {np.abs(mine['seasonal'] - sm.seasonal.to_numpy()).max():.1e}；"
          f"殘差差 {np.nanmax(np.abs(mine['resid'] - sm.resid.to_numpy())):.1e}")
t_idx = np.arange(len(x_train), dtype=float)
for frac in [0.05, 0.15, 0.4]:
    print(f"✅ LOESS frac = {frac}：跟 statsmodels lowess(it=0) 的最大差異 {np.abs(loess(t_idx, x_train.to_numpy(), frac) - lowess(x_train.to_numpy(), t_idx, frac=frac, it=0, return_sorted=False)).max():.1e}")

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(t_idx, x_train.to_numpy(), color="k", lw=0.8, label="log y")
for frac in [0.05, 0.15, 0.4]:
    axes[0].plot(t_idx, loess(t_idx, x_train.to_numpy(), frac), label=f"LOESS frac={frac}")
axes[0].legend()
dec_log = classical_decompose(x_train.to_numpy(), m=M)
axes[1].bar(np.arange(1, 13), 100 * (np.exp(dec_log["s_idx"]) - 1))
axes[1].set_xlabel("month")
axes[1].set_ylabel("% vs yearly level")
axes[1].set_title("seasonal index (log decomposition → %)")
plt.show()

In [ ]:
# TODO 7（問答）TODO 3–6 與實驗
#
# (1) 手寫的古典分解、LOESS 跟 statsmodels 差多少？
# (2) LOESS：frac = 0.05 的曲線為什麼會跟著季節上下跳？frac 要多大才「看不到季節」？（提示：一個季節 12 個月，n = 120 → frac ≈ ?）
# (3) 季節指數：哪幾個月比全年平均高最多？大約高幾 %？最低的月份低幾 %？
#     log 尺度的季節指數 0.2 代表比平均高多少 %？為什麼 exp(0.2) − 1 ≠ 20%？
# 答：

---
## 4. 套件實作：statsmodels `STL`

`STL(x, period=12, seasonal=7, trend=None, robust=False).fit()`：

| 參數 | 意思 |
| --- | --- |
| `seasonal` | 季節平滑的窗口（奇數，**以「年」為單位**：每個月的子序列用前後幾年平滑）。小 → 季節可以每年變；大 → 季節幾乎固定（接近古典分解） |
| `trend` | 趨勢平滑的窗口（預設由 `seasonal` 與 `period` 決定） |
| `robust` | 穩健迭代（異常值降權） |

實驗：
1. `seasonal` = 7 vs 35（大到接近固定季節）：季節成分會怎麼變？
2. **注入異常值**：把 1955 年 7 月改成原來的 1.5 倍，比較 `robust=False／True` 的分解。
3. 趨勢與季節的強度 $F_T$、$F_S$（前置複習 4）。

In [ ]:
from statsmodels.tsa.seasonal import STL

stl7 = STL(x_train, period=M, seasonal=7).fit()
stl35 = STL(x_train, period=M, seasonal=35).fit()
fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
axes[0].plot(x_train.index, x_train, color="k", lw=0.8)
axes[0].plot(x_train.index, stl7.trend, label="trend (seasonal=7)")
axes[0].legend()
axes[1].plot(x_train.index, stl7.seasonal, label="seasonal=7")
axes[1].plot(x_train.index, stl35.seasonal, label="seasonal=35")
axes[1].plot(x_train.index, dec_log["seasonal"], ls=":", color="k", label="classical")
axes[1].legend(fontsize=8)
axes[2].plot(x_train.index, stl7.resid, label="STL (7) remainder")
axes[2].legend()
plt.tight_layout()
plt.show()


def strength(dec_trend: np.ndarray, dec_seasonal: np.ndarray, dec_resid: np.ndarray) -> dict:
    ok = ~np.isnan(dec_trend) & ~np.isnan(dec_resid)
    R, T, S = dec_resid[ok], dec_trend[ok], dec_seasonal[ok]
    return {"F_trend": max(0.0, 1 - R.var() / (T + R).var()), "F_seasonal": max(0.0, 1 - R.var() / (S + R).var()), "remainder std": R.std()}


display(pd.DataFrame({
    "classical": strength(dec_log["trend"], dec_log["seasonal"], dec_log["resid"]),
    "STL seasonal=7": strength(stl7.trend.to_numpy(), stl7.seasonal.to_numpy(), stl7.resid.to_numpy()),
    "STL seasonal=35": strength(stl35.trend.to_numpy(), stl35.seasonal.to_numpy(), stl35.resid.to_numpy()),
}).T.round(4))

In [ ]:
# 注入異常值：1955-07 × 1.5
x_out = x_train.copy()
k_out = x_out.index.get_loc(pd.Timestamp("1955-07-01"))
x_out.iloc[k_out] += np.log(1.5)
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
rows = {}
for robust in [False, True]:
    f = STL(x_out, period=M, seasonal=7, robust=robust).fit()
    axes[0].plot(x_out.index, f.seasonal - stl7.seasonal, label=f"robust={robust}")
    axes[1].plot(x_out.index, f.resid, label=f"robust={robust}")
    rows[f"robust={robust}"] = {"remainder at outlier": f.resid.iloc[k_out], "max |Δ seasonal| elsewhere": float(np.abs((f.seasonal - stl7.seasonal).drop(x_out.index[k_out])).max())}
axes[0].set_title("change in seasonal component caused by the outlier")
axes[1].set_title("remainder")
for ax in axes:
    ax.legend()
plt.show()
display(pd.DataFrame(rows).T.round(4))
print("注入的大小（log）：", round(np.log(1.5), 4))

In [ ]:
# TODO 8（問答）STL
#
# (1) seasonal = 7 vs 35：季節成分的形狀隨年份變化嗎？哪一個比較接近古典分解？哪一個殘差比較小？殘差小一定比較好嗎？（提示：過度配適 —— 季節吸收了本該是殘差的東西）
# (2) 強度：F_trend、F_seasonal 各多少？這份資料是「趨勢主導」還是「季節主導」？
# (3) 異常值：
#     (a) 不穩健時，異常值的影響「漏」到了哪些月份的季節成分？為什麼？（提示：7 月的子序列用前後幾年平滑）
#     (b) 穩健時，異常點的殘差有多大？跟注入的大小 log 1.5 ≈ 0.405 比？—— 穩健的 STL 可以拿來做異常偵測
# 答：

---
## 5. 用分解來預測；調參

**分解預測**（Hyndman：「decomposition forecasting」）：

1. 分解 $x_t = S_t + A_t$，$A_t = x_t - S_t$ 是**季節調整後**的序列；
2. **季節**：用最後一年的季節值往後重複（季節性 naive）；
3. **季節調整後**：用一個非季節的方法預測 —— 這裡用 **drift**（最後一個值 + 平均斜率，第 1 章）或 ETS（Holt，statsmodels `STLForecast`）；
4. 相加，再 exp。

**TODO 9**：手寫「季節 naive + drift」版本。

候選（滾動起點 CV，同第 1、2 章：起點 84、96、108，各 12 步）：

| 候選 | 分解 | 季節調整後的預測 |
| --- | --- | --- |
| classical + drift（手寫） | 古典（季節指數固定） | drift |
| STL(7) + drift | STL `seasonal=7` | drift |
| STL(7) + ETS（`STLForecast`） | STL `seasonal=7` | Holt（加法趨勢） |
| STL(35) + ETS | STL `seasonal=35` | Holt |
| 季節性 naive | — | — |

#### 最小例子：TODO 9（$m = 2$）

季節調整後 $A = (10, 11, 12, 13)$（drift = 每步 +1）、最後一年的季節 $(+1, -1)$（資料的第 4 個點會是「第 0 個季節位置」）→ $h = 1, 2, 3$：$A$ 的預測 $14, 15, 16$，季節 $+1, -1, +1$ → $15, 14, 17$。

In [ ]:
def decomposition_forecast(x: np.ndarray, seasonal: np.ndarray, h: int, m: int = M) -> np.ndarray:
    """季節 naive + drift。x、seasonal (n,)：序列與它的季節成分（同一個尺度）。回傳 (h,) 的預測。"""
    n = len(x)
    adj = x - seasonal  # 季節調整後
    steps = np.arange(1, h + 1)
    # TODO 9（程式）：
    #   - adj_fc = adj[−1] + steps · (adj[−1] − adj[0]) / (n − 1)        ← drift
    #   - season_fc = seasonal[n − m + (steps − 1) % m]                  ← 最後一年的季節值往後重複
    #   - 回傳 adj_fc + season_fc
    raise NotImplementedError("TODO 9：分解預測")


# 寫完 TODO 9 之後，取消下面的註解來對照（應為 [15. 14. 17.]）：
# print(decomposition_forecast(np.array([11.0, 10.0, 13.0, 12.0]), np.array([1.0, -1.0, 1.0, -1.0]), 3, m=2))

In [ ]:
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.exponential_smoothing.ets import ETSModel


def rolling_origin_cv(y: pd.Series, make_forecast, origins: list[int], horizon: int = 12, m: int = M) -> pd.DataFrame:
    """第 1 章的版本（已完成）。"""
    rows = []
    for origin in origins:
        tr, va = y.iloc[:origin], y.iloc[origin : origin + horizon]
        rows.append({"origin": str(y.index[origin].date()), **forecast_metrics(va, make_forecast(tr, horizon), tr, season=m)})
    return pd.DataFrame(rows).set_index("origin")


def classical_drift(tr: pd.Series, h: int) -> np.ndarray:
    xt = np.log(tr.to_numpy())
    return np.exp(decomposition_forecast(xt, classical_decompose(xt)["seasonal"], h))


def stl_drift(seasonal_win: int):
    def f(tr: pd.Series, h: int) -> np.ndarray:
        xt = np.log(tr)
        return np.exp(decomposition_forecast(xt.to_numpy(), STL(xt, period=M, seasonal=seasonal_win).fit().seasonal.to_numpy(), h))
    return f


def stl_ets(seasonal_win: int):
    def f(tr: pd.Series, h: int) -> np.ndarray:
        xt = np.log(tr).asfreq("MS")
        res = STLForecast(xt, ETSModel, model_kwargs={"trend": "add"}, period=M, seasonal=seasonal_win).fit(fit_kwargs={"disp": False})
        return np.exp(res.forecast(h).to_numpy())
    return f


ORIGINS = [84, 96, 108]
candidates = {
    "snaive": lambda tr, h: tr.to_numpy()[len(tr) - M + (np.arange(h) % M)],
    "classical + drift (scratch)": classical_drift,
    "STL(7) + drift": stl_drift(7),
    "STL(7) + ETS": stl_ets(7),
    "STL(35) + ETS": stl_ets(35),
}
N_PARAMS = {"snaive": 0, "classical + drift (scratch)": 1, "STL(7) + drift": 1, "STL(7) + ETS": 3, "STL(35) + ETS": 3}
cv_rows = {}
for name, f in candidates.items():
    cv = rolling_origin_cv(y_train, f, ORIGINS)
    cv_rows[name] = {"mean MASE": cv["mase"].mean(), "std MASE": cv["mase"].std(), **{f"MASE @{o}": v for o, v in cv["mase"].items()}}
cv_table = pd.DataFrame(cv_rows).T.sort_values("mean MASE")
display(cv_table.round(3))
chosen = cv_table.index[0]
print("CV mean MASE 最低：", chosen, "（第 2 章的教訓：只有 3 個 fold，one-standard-error rule 太寬鬆，這裡直接選最低的）")

In [ ]:
# TODO 10（問答）
#
# (1) CV 的排名：分解 + drift 比季節性 naive 好多少？STL 比古典分解好嗎？drift 與 ETS（Holt）誰好？
# (2) 「季節 naive + drift」的 drift 是用「整段的平均斜率」—— 跟 Holt 的「最近的斜率」比，各適合什麼情況？
# (3) 為什麼分解預測要在 log 尺度上做？如果在原尺度用加法分解，最後一年的季節值放到未來會有什麼問題？
# 答：

---
## 6. 測試集評估

選出的方法在整個 `y_train` 上分解、預測 test 24 個月，只評估一次。也放進其他分解方法當參照。成績接著寫進 `results/air_passengers.csv`（跟第 1、2 章一起比）。

In [ ]:
from ml_dl_llm.results import compare, log_result

BENCH = "air_passengers"
fc_all = {name: f(y_train, H) for name, f in candidates.items() if name != "snaive"}
for name, fc in fc_all.items():
    label = f"{name} (CV-selected)" if name == chosen else name
    test_table.loc[label] = forecast_metrics(y_test, fc, y_train, season=M)
    log_result(BENCH, label, test_table.loc[label].to_dict(), horizon=H)
display(test_table.round(4))
compare(BENCH, "mase", higher_is_better=False)

---
## 7. 診斷

1. 選出的方法的預測 vs 實際，以及「季節調整後」的序列在 test 期間的走勢（趨勢預測對不對？）。
2. 季節成分的預測誤差：test 期間「實際的季節形狀」（用整條序列重新分解）vs 預測用的「最後一年的季節」。

In [ ]:
fc_best = fc_all[chosen]
x_full = np.log(y)
stl_full = STL(x_full, period=M, seasonal=7).fit()
stl_tr = STL(x_train, period=M, seasonal=7).fit()
adj_full = x_full - stl_full.seasonal
fig, axes = plt.subplots(1, 3, figsize=(19, 3.8))
axes[0].plot(y.iloc[-48:].index, y.iloc[-48:], color="k", label="actual")
axes[0].plot(y_test.index, fc_best, color="tab:red", label=chosen)
axes[0].legend(fontsize=8)
axes[1].plot(adj_full.index[-60:], adj_full.iloc[-60:], color="k", label="seasonally adjusted (full-data STL)")
adj_tr = x_train - stl_tr.seasonal
slope = (adj_tr.iloc[-1] - adj_tr.iloc[0]) / (len(adj_tr) - 1)
axes[1].plot(y_test.index, adj_tr.iloc[-1] + slope * np.arange(1, H + 1), color="tab:red", ls="--", label="drift forecast")
axes[1].legend(fontsize=8)
axes[2].plot(np.arange(1, 13), stl_tr.seasonal.iloc[-M:].to_numpy(), "o-", label="last train year (used)")
axes[2].plot(np.arange(1, 13), stl_full.seasonal.iloc[-24:-12].to_numpy(), "o-", label="1959 (full-data STL)")
axes[2].plot(np.arange(1, 13), stl_full.seasonal.iloc[-12:].to_numpy(), "o-", label="1960 (full-data STL)")
axes[2].legend(fontsize=8)
axes[2].set_xlabel("month")
plt.show()
pct_err = pd.Series(100 * (fc_best / y_test.to_numpy() - 1), index=y_test.index.strftime("%Y-%m"))
print("每個月的百分比誤差：", pct_err.round(1).to_dict())

In [ ]:
# TODO 11（問答，本章總結）
#
# (1) 測試集：分解預測的 MASE 是多少？在第 1、2 章的排行榜上排第幾？
# (2) 診斷：誤差主要來自「趨勢的預測」（中圖）還是「季節形狀的變化」（右圖）？1959–1960 年的季節形狀跟 1958 年差多少？
# (3) 分解 vs 第 1 章的 Holt-Winters：兩者都把序列拆成趨勢 + 季節，差別在哪裡？（提示：中心化 vs 只用過去；估計一次 vs 一步一步更新）
# (4) 分解的其他用途：季節調整（官方統計）、異常偵測（實驗：穩健 STL 的殘差）。你能想到一個「需要季節調整後的數字」的例子嗎？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

AirPassengers（train 1949–1958、test 1959–1960，24 步預測；log 尺度分解）

| 方法 | 分解 | 季節調整後的預測 | CV mean MASE | test MASE |
| --- | --- | --- | --- | --- |
| 季節性 naive | — | — | | |
| classical + drift（手寫） | | | | |
| STL(7) + drift | | | | |
| STL(7)／STL(35) + ETS | | | | |
| （第 1 章）log + Holt-Winters 加法 | — | — | 0.61 | 0.66 |
| （第 2 章）航空模型 | — | — | | 1.38 |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ STL 的完整演算法（內迴圈：子序列平滑、低通濾波、趨勢平滑；外迴圈：穩健權重）
- ☐ LOESS 的頻寬選擇、局部多項式的階數
- ☐ X-11、X-13ARIMA-SEATS（官方統計的季節調整）

**延伸**
- ☐ MSTL：多重季節（日資料的週 + 年）
- ☐ 用分解做異常偵測（穩健 STL 殘差 + 門檻）
- ☐ 第 7 章 Prophet：另一種「趨勢 + 季節 + 節日」的加法模型